# Return Risk Model — Myntra Fashion
### Deliverable 1: Feature spec → p(return) → decomposition → intervention routing

**What this notebook does:** At checkout, we score each order for return probability using 11 signals. If the score crosses a threshold, we decompose it to find the top contributing feature, then route to the right intervention — not a blanket policy, but a targeted nudge based on what's actually causing the risk.

**Flow:** features → logistic regression → p(return) → if p > threshold → decompose → top driver → intervention

In [41]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')


## 1. Features list used as signals

11 signals collected at checkout. Each maps to a data source we already have.

**Three groups:**
- **User history** — what this buyer has done before. Strongest signals but unavailable for new users.
- **Item features** — properties of what they're buying. Category return rate and size confidence are the most predictive.
- **Context** — how and where they're buying. COD is the single biggest contextual risk flag.

| Feature | Rationale |
|---|---|
| `user_return_rate_90d` | Recency-weighted return behaviour. Strongest individual signal. |
| `user_return_rate_1yr` | Long-run baseline. Corrects for seasonal noise in 90d window. |
| `user_order_count` | Cold-start weight. Low count → regularise toward category base rate. |
| `category_base_rate` | Structural prior. Swimwear ~35%, accessories ~8%. Model starts here. |
| `size_selection_confidence` | Did user consult + follow size guide? Low = uncertain = high risk. |
| `size_changed_in_session` | Swapped size in cart. Each swap ≈ +3pp return probability. |
| `price_band` | 1=<₹500, 2=₹500–3000, 3=>₹3000. High price → deliberate purchase → lower risk. |
| `payment_method_cod` | COD returns at ~2x prepaid rate. Zero skin in the game at order time. |
| `device_type` | 0=mobile web, 1=desktop, 2=app. App = higher intent user. |
| `is_first_order` | No history. Cold-start flag — regularise to category base rate. |
| `images_viewed_count` | More images viewed = more deliberate purchase = lower return risk. |

In [42]:
feature_registry = pd.DataFrame([
    ("user_return_rate_90d",      "float 0-1",  "orders DB",        "user_history"),
    ("user_return_rate_1yr",      "float 0-1",  "orders DB",        "user_history"),
    ("user_order_count",          "int",        "orders DB",        "user_history"),
    ("category_base_rate",        "float 0-1",  "catalogue DB",     "item"),
    ("size_selection_confidence", "float 0-1",  "size tool events", "item"),
    ("size_changed_in_session",   "bool 0/1",   "session events",   "item"),
    ("price_band",                "int 1-3",    "catalogue DB",     "item"),
    ("payment_method_cod",        "bool 0/1",   "payments DB",      "context"),
    ("device_type",               "int 0-2",    "session events",   "context"),
    ("is_first_order",            "bool 0/1",   "orders DB",        "context"),
    ("images_viewed_count",       "int",        "session events",   "context"),
], columns=["feature", "type", "source", "group"])

feature_registry

,feature,type,source,group
0,user_return_rate_90d,float 0-1,orders DB,user_history
1,user_return_rate_1yr,float 0-1,orders DB,user_history
2,user_order_count,int,orders DB,user_history
3,category_base_rate,float 0-1,catalogue DB,item
4,size_selection_confidence,float 0-1,size tool events,item
5,size_changed_in_session,bool 0/1,session events,item
6,price_band,int 1-3,catalogue DB,item
7,payment_method_cod,bool 0/1,payments DB,context
8,device_type,int 0-2,session events,context
9,is_first_order,bool 0/1,orders DB,context


## 2. Synthetic Training Data

Generates 10,000 fake orders to stand in for a real orders table. The return label is constructed from domain logic — COD, low size confidence, high category rate push probability up; high price, app usage, high image views push it down.

**To use with real data:** replace the  block with a SQL query that pulls from your orders + events tables. Everything downstream is the same.

**Why 10,000?** Logistic regression needs roughly 10–20x the number of features as training examples to be stable. With 11 features, 10k is more than sufficient. For a production model on real data you'd have millions.

In [43]:
np.random.seed(42)
N = 10_000

df = pd.DataFrame({
    "user_return_rate_90d":      np.clip(np.random.beta(2, 5, N), 0, 1),
    "user_return_rate_1yr":      np.clip(np.random.beta(2, 6, N), 0, 1),
    "user_order_count":          np.random.randint(1, 80, N),
    "category_base_rate":        np.clip(np.random.beta(3, 7, N), 0.05, 0.45),
    "size_selection_confidence": np.clip(np.random.beta(5, 3, N), 0, 1),
    "size_changed_in_session":   np.random.binomial(1, 0.25, N),
    "price_band":                np.random.choice([1, 2, 3], N, p=[0.3, 0.5, 0.2]),
    "payment_method_cod":        np.random.binomial(1, 0.45, N),
    "device_type":               np.random.choice([0, 1, 2], N, p=[0.35, 0.15, 0.50]),
    "is_first_order":            np.random.binomial(1, 0.20, N),
    "images_viewed_count":       np.random.randint(1, 12, N),
})

# Synthetic label — encodes domain logic directionally
log_odds = (
     2.0 * df.user_return_rate_90d
   + 1.0 * df.user_return_rate_1yr
   + 1.5 * df.category_base_rate
   - 1.5 * df.size_selection_confidence
   + 0.6 * df.size_changed_in_session
   - 0.4 * df.price_band
   + 1.2 * df.payment_method_cod
   - 0.3 * df.device_type
   + 0.4 * df.is_first_order
   - 0.1 * df.images_viewed_count
   - 1.5  # intercept
)
prob = 1 / (1 + np.exp(-log_odds))
df["returned"] = np.random.binomial(1, prob)

print(f"Return rate in dataset: {df.returned.mean():.1%}")
print(f"Shape: {df.shape}")
df.head(3)


Return rate in dataset: 14.2%
Shape: (10000, 12)


,user_return_rate_90d,user_return_rate_1yr,user_order_count,category_base_rate,size_selection_confidence,size_changed_in_session,price_band,payment_method_cod,device_type,is_first_order,images_viewed_count,returned
0,0.353677,0.421414,11,0.450000,0.728031,0,2,1,0,1,10,1
1,0.248558,0.183193,67,0.278023,0.934960,1,1,0,0,1,4,0
2,0.415959,0.062865,58,0.450000,0.556572,0,1,1,0,0,10,0


## 3. Train Logistic Regression → p(return)

**What StandardScaler does:** Normalises each feature to mean=0, std=1. This is required because logistic regression is sensitive to feature scale — without it,  (range 1–12) would be treated as less important than  (range 0–1) purely because of units, not signal strength.

**Critical:** fit the scaler only on training data (), then apply the same scaler to test data and new orders ( only). If you fit on all data you're leaking future information into training.

**AUC-ROC** is the right metric here, not accuracy. It measures how well the model separates returners from non-returners across all thresholds. Above 0.70 is usable; above 0.75 is good for this type of problem.

In [44]:
FEATURE_COLS = [
    "user_return_rate_90d", "user_return_rate_1yr", "user_order_count",
    "category_base_rate", "size_selection_confidence", "size_changed_in_session",
    "price_band", "payment_method_cod", "device_type",
    "is_first_order", "images_viewed_count"
]

X = df[FEATURE_COLS].values
y = df["returned"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LogisticRegression(class_weight='balanced',max_iter=1000)
model.fit(X_train_s, y_train)

y_pred_proba = model.predict_proba(X_test_s)[:, 1]
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred_proba):.3f}")
print()
print(classification_report(y_test, (y_pred_proba > 0.5).astype(int)))


AUC-ROC: 0.767

              precision    recall  f1-score   support

           0       0.94      0.67      0.78      1741
           1       0.25      0.73      0.37       259

    accuracy                           0.67      2000
   macro avg       0.59      0.70      0.57      2000
weighted avg       0.85      0.67      0.73      2000



## 4. Feature Coefficients — global importance

The model assigns one coefficient to each feature. After scaling, these are directly comparable — larger absolute value = stronger influence on the score.

**Positive coefficient** → feature increases return probability when it goes up.  
**Negative coefficient** → feature decreases return probability when it goes up.

This table is your first validation step. If  doesn't appear near the top with a positive coefficient, something is wrong — either the data encoding is off, or the training data has a confound. Fix the data before tuning the model.

This is also what you'd show to legal or compliance if they ask why a feature is included.

In [45]:
coef_df = pd.DataFrame({
    "feature":     FEATURE_COLS,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", ascending=False).reset_index(drop=True)

coef_df["direction"] = coef_df.coefficient.apply(
    lambda x: "↑ increases risk" if x > 0 else "↓ decreases risk"
)

coef_df


,feature,coefficient,direction
0,payment_method_cod,0.537281,↑ increases risk
1,user_return_rate_90d,0.269954,↑ increases risk
2,size_changed_in_session,0.205502,↑ increases risk
3,category_base_rate,0.155258,↑ increases risk
4,user_return_rate_1yr,0.140364,↑ increases risk
5,is_first_order,0.129665,↑ increases risk
6,user_order_count,0.004636,↑ increases risk
7,price_band,-0.243723,↓ decreases risk
8,images_viewed_count,-0.245351,↓ decreases risk
9,size_selection_confidence,-0.257936,↓ decreases risk


## 5. Score a Single Order → p(return)

This is the function that runs at checkout — takes one order's features, returns one number between 0 and 1.

**What's happening internally:**
1. Scale the 11 features using the same scaler fitted on training data
2. Multiply each scaled feature by its learned coefficient
3. Sum everything + intercept = log-odds
4. Pass log-odds through sigmoid = probability

The example order is deliberately high-risk: high return history, low size confidence, COD, size switch, few images viewed. You should see a score above 0.5.

In [46]:
def score_order(order_dict):
    """
    Input:  dict with all 11 feature values
    Output: float p(return) between 0 and 1
    """
    values = np.array([[order_dict[f] for f in FEATURE_COLS]])
    scaled = scaler.transform(values)
    return model.predict_proba(scaled)[0, 1]


# Example order
example_order = {
    "user_return_rate_90d":      0.55,
    "user_return_rate_1yr":      0.40,
    "user_order_count":          8,
    "category_base_rate":        0.35,   # e.g. dresses
    "size_selection_confidence": 0.30,   # low — user ignored size guide
    "size_changed_in_session":   1,      # switched sizes
    "price_band":                1,      # sub ₹500
    "payment_method_cod":        1,      # COD
    "device_type":               0,      # mobile web
    "is_first_order":            0,
    "images_viewed_count":       2,
}

p_return = score_order(example_order)
print(f"p(return) = {p_return:.3f}")


p(return) = 0.943


## 6. Decomposition — which feature is the culprit?

A score of 0.73 doesn't tell you what to do. Two orders can score identically via completely different paths:

- Order A: high COD + high return history → show prepaid incentive  
- Order B: low size confidence + size switch → show size guide gate

Decomposition pulls apart the summed log-odds into per-feature contributions. For a specific order, each feature's contribution is its scaled value multiplied by its learned coefficient. Positive contributions are pushing the score up — those are the risk drivers.

The feature with the highest positive contribution is the top driver. The intervention routing logic uses the top driver — not the score — to decide what to show the user.

In [47]:
THRESHOLD = 0.6  # decompose only if p(return) exceeds this

def decompose(order_dict, threshold=THRESHOLD):
    """
    Returns (score, culprits_df, top_driver) if score >= threshold.
    Returns (score, None, None) if below threshold.
    """
    values = np.array([[order_dict[f] for f in FEATURE_COLS]])
    scaled = scaler.transform(values)
    
    contributions = scaled[0] * model.coef_[0]
    log_odds_val  = model.intercept_[0] + contributions.sum()
    score         = float(1 / (1 + np.exp(-log_odds_val)))
    
    if score < threshold:
        return score, None, None
    
    culprits = (
        pd.DataFrame({"feature": FEATURE_COLS, "contribution": contributions})
        .query("contribution > 0")
        .sort_values("contribution", ascending=False)
        .reset_index(drop=True)
    )
    culprits["contribution_%"] = (
        culprits.contribution / culprits.contribution.sum() * 100
    ).round(1)
    
    top_driver = culprits.iloc[0]["feature"]
    return score, culprits, top_driver


score, culprits, top_driver = decompose(example_order)

print(f"p(return) = {score:.3f}  |  threshold = {THRESHOLD}")
print(f"Top driver: {top_driver}\n")
culprits


p(return) = 0.943  |  threshold = 0.6
Top driver: payment_method_cod



,feature,contribution,contribution_%
0,payment_method_cod,0.584920,18.8
1,size_selection_confidence,0.518926,16.7
2,user_return_rate_90d,0.456250,14.7
3,size_changed_in_session,0.359777,11.6
4,device_type,0.334690,10.8
5,images_viewed_count,0.311588,10.0
6,price_band,0.310057,10.0
7,user_return_rate_1yr,0.146895,4.7
8,category_base_rate,0.082264,2.6


## 7. Intervention Table

Maps each feature to one intervention. The logic: the feature that's most responsible for the high score should determine what we show the user.

**Design principle:** interventions should address the cause of risk, not just punish it. A buyer with low size confidence needs a size guide, not a COD restriction. A chronic returner who's already using the size guide needs an exchange-first offer.

 rows exist because some features (device_type, user_order_count) are scoring signals only — there's no sensible user-facing action tied to them. The routing function automatically falls back to the second-highest driver in these cases.

In [48]:
INTERVENTION_TABLE = pd.DataFrame([
    # feature,                      intervention_type,           message_to_user
    ("payment_method_cod",
     "prepaid_incentive",
     "Pay now and get free size exchange + 10% credit on next order."),

    ("size_selection_confidence",
     "size_guide_gate",
     "You skipped the size guide. 3 in 10 customers return this category — takes 30s."),

    ("size_changed_in_session",
     "size_confirmation_nudge",
     "You changed your size. Want to re-check using the size guide before confirming?"),

    ("user_return_rate_90d",
     "exchange_first_offer",
     "If this doesn't fit, exchange for the right size — free pickup, no questions asked."),

    ("user_return_rate_1yr",
     "exchange_first_offer",
     "If this doesn't fit, exchange for the right size — free pickup, no questions asked."),

    ("category_base_rate",
     "category_size_tip",
     "Sizing in this category runs small. Most buyers go one size up."),

    ("price_band",
     "no_intervention",
     "Low price band — impulse purchase risk. No friction justified at this price."),

    ("is_first_order",
     "size_guide_gate",
     "First order? Our size guide takes 30s and halves the chance you'll need to return."),

    ("images_viewed_count",
     "product_detail_nudge",
     "You haven't seen all product images. Check the size chart photo before confirming."),

    ("device_type",
     "no_intervention",
     "Device signal — weak standalone driver, no user-facing intervention."),

    ("user_order_count",
     "no_intervention",
     "Cold-start signal — used for scoring only, no user-facing intervention."),

], columns=["feature", "intervention_type", "user_message"])

INTERVENTION_TABLE


,feature,intervention_type,user_message
0,payment_method_cod,prepaid_incentive,Pay now and get free size exchange + 10% credi...
1,size_selection_confidence,size_guide_gate,You skipped the size guide. 3 in 10 customers ...
2,size_changed_in_session,size_confirmation_nudge,You changed your size. Want to re-check using ...
3,user_return_rate_90d,exchange_first_offer,"If this doesn't fit, exchange for the right si..."
4,user_return_rate_1yr,exchange_first_offer,"If this doesn't fit, exchange for the right si..."
5,category_base_rate,category_size_tip,Sizing in this category runs small. Most buyer...
6,price_band,no_intervention,Low price band — impulse purchase risk. No fri...
7,is_first_order,size_guide_gate,First order? Our size guide takes 30s and halv...
8,images_viewed_count,product_detail_nudge,You haven't seen all product images. Check the...
9,device_type,no_intervention,"Device signal — weak standalone driver, no use..."


## 8. Intervention Routing — full pipeline

Combines everything into one function: order dict goes in, intervention decision comes out.

**Two axes working together**
- Score band answers "how hard do we push?" — severity of intervention
- Top driver answers "what do we say?" — type of intervention

Neither alone is sufficient. Same score can have different causes. Same cause can warrant different severity depending on score.

```
score < 0.40              → no intervention
score 0.40 – 0.70         → soft: nudge only, type set by top driver
score 0.70 – 0.85         → medium: nudge + COD friction if COD is in top 3 drivers
score > 0.85              → hard: COD restricted regardless of top driver, always show alternative
```

**Fallback logic:** if the top driver maps to `no_intervention` (e.g. `device_type`), the function walks down the culprits list to find the first driver that has a real intervention. This prevents the model from firing on a weak signal when a stronger, actionable signal is also present.

**Ethical guardrail built into the hard band:** COD is never silently blocked. The user always sees a reason and an alternative (prepaid + free exchange incentive). This is non-negotiable — silent blocks generate complaints and chargebacks.

**What to check before rollout:**
- `score`, `score_band`, `top_driver`, `intervention_type` for every order
- No changes visible to the user yet
- After 2 weeks: check if high-score orders are returning at the predicted rate. If yes, the model is calibrated and the interventions can be rolled out

In [49]:
def get_score_band(score):
    """
    Severity tier based on score alone.
    Answers: how hard do we push?
    """
    if score < 0.40:
        return "none"
    elif score < 0.70:
        return "soft"
    elif score < 0.85:
        return "medium"
    else:
        return "hard"


def route_intervention(order_dict, threshold=THRESHOLD):
    """
    Full pipeline: order → score → score band → decompose → intervention.

    Score band sets severity. Top driver sets type.
    Returns a dict with score, score_band, top_driver, intervention_type, message.
    """
    score, culprits, top_driver = decompose(order_dict, threshold)

    score_band = get_score_band(score)

    # No intervention below threshold
    if score_band == "none":
        return {
            "score":             round(score, 3),
            "score_band":        score_band,
            "action":            "no_intervention",
            "top_driver":        None,
            "intervention_type": "none",
            "user_message":      "Checkout proceeds normally.",
            "culprits":          None,
        }

    # Hard band: COD restriction regardless of top driver
    if score_band == "hard":
        return {
            "score":             round(score, 3),
            "score_band":        score_band,
            "action":            "intervene",
            "top_driver":        top_driver,
            "intervention_type": "restrict_cod",
            "user_message":      "Pay now to unlock free size exchange + 10% credit on next order. COD unavailable for this order.",
            "culprits":          culprits,
        }

    # Medium band: nudge by top driver + add COD friction if COD is in top 3 drivers
    if score_band == "medium":
        cod_in_top3 = (
            culprits is not None and
            "payment_method_cod" in culprits.head(3)["feature"].values
        )

        # Get type-based nudge from top driver (same fallback logic as before)
        row = INTERVENTION_TABLE[INTERVENTION_TABLE.feature == top_driver]
        if row.empty or row.iloc[0].intervention_type == "no_intervention":
            for _, candidate in culprits.iterrows():
                row2 = INTERVENTION_TABLE[INTERVENTION_TABLE.feature == candidate.feature]
                if not row2.empty and row2.iloc[0].intervention_type != "no_intervention":
                    row = row2
                    top_driver = candidate.feature
                    break

        intervention_type = row.iloc[0].intervention_type if not row.empty else "none"
        user_message      = row.iloc[0].user_message      if not row.empty else ""

        if cod_in_top3:
            intervention_type = intervention_type + " + cod_friction"
            user_message      = user_message + " Prepaid unlocks free size exchange."

        return {
            "score":             round(score, 3),
            "score_band":        score_band,
            "action":            "intervene",
            "top_driver":        top_driver,
            "intervention_type": intervention_type,
            "user_message":      user_message,
            "culprits":          culprits,
        }

    # Soft band: nudge only, type set by top driver
    row = INTERVENTION_TABLE[INTERVENTION_TABLE.feature == top_driver]
    if row.empty or row.iloc[0].intervention_type == "no_intervention":
        for _, candidate in culprits.iterrows():
            row2 = INTERVENTION_TABLE[INTERVENTION_TABLE.feature == candidate.feature]
            if not row2.empty and row2.iloc[0].intervention_type != "no_intervention":
                row = row2
                top_driver = candidate.feature
                break

    intervention_type = row.iloc[0].intervention_type if not row.empty else "none"
    user_message      = row.iloc[0].user_message      if not row.empty else ""

    return {
        "score":             round(score, 3),
        "score_band":        score_band,
        "action":            "intervene",
        "top_driver":        top_driver,
        "intervention_type": intervention_type,
        "user_message":      user_message,
        "culprits":          culprits,
    }


result = route_intervention(example_order)

print(f"Score:             {result['score']}")
print(f"Score band:        {result['score_band']}")
print(f"Action:            {result['action']}")
print(f"Top driver:        {result['top_driver']}")
print(f"Intervention:      {result['intervention_type']}")
print(f"Message to user:   {result['user_message']}")
print()
print("Full decomposition:")
result["culprits"]


Score:             0.943
Score band:        hard
Action:            intervene
Top driver:        payment_method_cod
Intervention:      restrict_cod
Message to user:   Pay now to unlock free size exchange + 10% credit on next order. COD unavailable for this order.

Full decomposition:


,feature,contribution,contribution_%
0,payment_method_cod,0.584920,18.8
1,size_selection_confidence,0.518926,16.7
2,user_return_rate_90d,0.456250,14.7
3,size_changed_in_session,0.359777,11.6
4,device_type,0.334690,10.8
5,images_viewed_count,0.311588,10.0
6,price_band,0.310057,10.0
7,user_return_rate_1yr,0.146895,4.7
8,category_base_rate,0.082264,2.6


## 9. Batch Scoring — distribution check

Before going live, you need to know how many orders will be affected by each intervention band. If 60% of orders are flagged, the threshold is too low and you'll tank conversion. If 2% are flagged, the threshold may be too high to have business impact.

**Rule of thumb:** interventions should touch 10–20% of orders at most. Use this distribution to calibrate your threshold before launch, not after.

The bands here mirror the intervention decision tree from Deliverable 2.

In [50]:
# Score all test orders
X_test_df = pd.DataFrame(X_test, columns=FEATURE_COLS)
X_test_df["p_return"] = model.predict_proba(X_test_s)[:, 1]

# Distribution across score bands
bands = pd.cut(
    X_test_df.p_return,
    bins=[0, 0.4, 0.55, 0.70, 0.85, 1.0],
    labels=["<0.40 no action", "0.40–0.55 size nudge",
            "0.55–0.70 exchange offer", "0.70–0.85 COD friction", ">0.85 restrict COD"]
)
print(bands.value_counts().sort_index().to_string())
print(f"\nOrders requiring intervention: {(X_test_df.p_return >= THRESHOLD).mean():.1%}")


p_return
<0.40 no action             886
0.40–0.55 size nudge        498
0.55–0.70 exchange offer    399
0.70–0.85 COD friction      203
>0.85 restrict COD           14

Orders requiring intervention: 22.9%


## Notes

| Item | Value |
|---|---|
| Model | Logistic Regression (sklearn) |
| Intervention threshold | 0.50 — tune based on conversion impact |
| Retraining cadence | Weekly batch on last 90d orders |
| Cold-start handling | `is_first_order=1` → regularise to category base rate |
| Excluded features | Pincode, device price proxy, return reason text (leakage risk) |
| Next step | Replace synthetic data with real orders table, validate AUC > 0.70 |
